# StatCore LDA: reliability-oriented discriminant analysis

[Open in Colab](https://colab.research.google.com/github/Jabualani/ML-Reliabilty/blob/main/docs/colab/statcore-lda.ipynb)

**Paper topic:** Reliability-Oriented Linear Discriminant Analysis in StatCore — `docs/latex/statcore-lda.tex` ([repository](https://github.com/Jabualani/ML-Reliabilty/blob/main/docs/latex/statcore-lda.tex)).
**Experiment script reproduced here:** `docs/latex/experiments/lda_experiment.py` (reduced seed count so the notebook runs in a few minutes).

StatCore is a pure-NumPy/SciPy model family in `ml_reliability.statcore`. Every model has a `variant`
argument; each variant below is compared with the standard reference implementation.
**scikit-learn is used only as the baseline/reference** (and for loading public datasets), never inside StatCore.

## 1. Install (parameter cell)

> **Note:** StatCore has **not been pushed / published yet**. Until the repository is public, the
> `git+https` install below will fail. In that case use the **fallback cell**: zip the repository folder,
> upload the zip to Colab (left sidebar → Files, or the `files.upload()` helper) and `pip install` it.
> Run exactly one of the two cells.

In [ ]:
# @title Install StatCore (PARAMETER: change the source if needed)
# Option A - once the repository is published:
!pip install "git+https://github.com/Jabualani/ML-Reliabilty.git"

In [ ]:
# @title FALLBACK install from an uploaded zip of the repository (use if Option A fails)
# Zip the repo folder (ML-Reliabilty.zip), then uncomment and run:
# from google.colab import files
# files.upload()                       # pick ML-Reliabilty.zip
# !unzip -q -o ML-Reliabilty.zip -d repo
# !pip install ./repo/ML-Reliabilty     # adjust to the folder that contains pyproject.toml

In [ ]:
# @title Parameters and imports
SEEDS = (11, 23, 37)   # the paper uses more seeds; fewer here to keep the run short

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## 2. Method summary

Data $(x_i,y_i)$, classes $k=1..C$, sample weights $w_i$ ($=1$ except `shift_corrected`), class weight $W_k=\sum_{i:y_i=k}w_i$.
Priors $\pi_k=W_k/\sum W$, means $\mu_k=\sum_{i:y_i=k}w_ix_i/W_k$, pooled within-class covariance
$$\Sigma=\frac{\sum_i w_i\,(x_i-\mu_{y_i})(x_i-\mu_{y_i})^\top}{\sum_i w_i}+\texttt{reg}\cdot\frac{\mathrm{tr}\,\Sigma}{d}I.$$
Linear score (all variants except `heteroscedastic`): $\delta_k(x)=x^\top\Sigma^{-1}\mu_k-\tfrac12\mu_k^\top\Sigma^{-1}\mu_k+\log\pi_k$; probabilities are $\mathrm{softmax}_k\,\delta_k(x)$.

**`base`** — the classic pooled LDA above (reference: sklearn `LinearDiscriminantAnalysis(solver="lsqr")`).

**`shift_corrected`** (covariate shift). A logistic domain classifier $g(x)=P(\text{target}\mid x)$ on standardised features (L2 = 1, Newton) gives
$$w_i=\min\!\Big(\tfrac{g(x_i)}{1-g(x_i)}\cdot\tfrac{n_s}{n_t},\,20\Big),\qquad w\leftarrow w\,n_s/\textstyle\sum w,$$
then weighted priors, means and covariance. `fit(X, y, X_target=...)` requires unlabeled target features.

**`shrinkage`** — $\Sigma_s=(1-s)\Sigma+s\,\frac{\mathrm{tr}\Sigma}{d}I$ with the analytic Ledoit–Wolf intensity $s=\min(\hat b^2,\hat\delta^2)/\hat\delta^2\in[0,1]$ (or fixed `shrinkage`).

**`sparse`** (nearest shrunken centroids). With $s_j=\sqrt{\Sigma_{jj}}$, $s_0=\mathrm{median}_j\,s_j$, $m_k=\sqrt{1/n_k-1/n}$:
$$d_{kj}=\frac{\mu_{kj}-\bar x_j}{m_k(s_j+s_0)},\quad d'_{kj}=\mathrm{sign}(d_{kj})\,(|d_{kj}|-\Delta)_+,\quad \mu'_{kj}=\bar x_j+m_k(s_j+s_0)d'_{kj},$$
with diagonal covariance $\mathrm{diag}(\Sigma)$; features with all $d'_{kj}=0$ are discarded (`selected_features_`).

**`heteroscedastic`** (regularised QDA). $\Sigma_k=(1-\rho)\hat\Sigma_k+\rho\Sigma_{\text{pooled}}+10^{-8}I$ with `reg_param`$=\rho$, and
$\delta_k(x)=-\tfrac12\big[(x-\mu_k)^\top\Sigma_k^{-1}(x-\mu_k)+\log|\Sigma_k|\big]+\log\pi_k$.

## 3. Experiment (mirrors `lda_experiment.py`)

Seven cases: `clean`, covariate `shifted` test set (the `shift_corrected` variant gets an unlabeled sample from the shifted domain; for other cases it receives a copy of the training features), 15 % `label_noise`, 8 % gross `outliers`, `high_dim` ($d=80$, $n=120$), `heteroscedastic` class scales, and the real `wine` data. Metrics: accuracy, log-loss and a 10-bin confidence ECE.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis   # baseline only
from ml_reliability.statcore import LDA
from ml_reliability.statcore.lda.lda import VARIANTS

CASES = ("clean", "shifted", "label_noise", "outliers", "high_dim", "heteroscedastic", "wine")

def stratified_split(y, fraction, seed):
    rng = np.random.default_rng(seed); tr, te = [], []
    for cls in np.unique(y):
        idx = np.flatnonzero(y == cls); rng.shuffle(idx)
        n_test = min(max(1, int(round(len(idx) * fraction))), len(idx) - 1)
        te.extend(idx[:n_test]); tr.extend(idx[n_test:])
    return np.array(sorted(tr)), np.array(sorted(te))

def synthetic(case, seed):
    rng = np.random.default_rng(seed)
    ntr, nte = (120, 300) if case == "high_dim" else (360, 600)
    d = 80 if case == "high_dim" else 12
    informative = 10 if case == "high_dim" else min(8, d)
    classes = 3
    centers = np.zeros((classes, d))
    centers[:, :informative] = np.array([
        np.r_[-2.0, 0.0, np.zeros(informative - 2)],
        np.r_[2.0, 0.0, np.zeros(informative - 2)],
        np.r_[0.0, 2.2, np.zeros(informative - 2)]])
    scale = np.ones((classes, d))
    if case == "heteroscedastic":
        scale[0, :informative] = 0.55; scale[1, :informative] = 1.8
    shift = np.zeros(d)
    if case == "shifted":
        shift[:informative] = np.linspace(0.5, 1.1, informative)
    def sample(n, offset):
        y = rng.integers(classes, size=n)
        return centers[y] + rng.normal(size=(n, d)) * scale[y] + offset, y
    xtr, ytr = sample(ntr, np.zeros(d)); xte, yte = sample(nte, shift)
    if case == "label_noise":
        noisy = rng.random(ntr) < 0.15
        ytr[noisy] = (ytr[noisy] + rng.integers(1, classes, noisy.sum())) % classes
    elif case == "outliers":
        out = rng.random(ntr) < 0.08
        xtr[out] = rng.normal(0, 6, size=(out.sum(), d))
    xadapt, _ = sample(500, shift)
    return xtr, ytr, xte, yte, (xadapt if case == "shifted" else None)

def dataset(case, seed):
    if case != "wine":
        return synthetic(case, seed)
    ds = load_wine(); tr, te = stratified_split(ds.target, 0.25, seed)
    return ds.data[tr], ds.target[tr], ds.data[te], ds.target[te], None

def metrics(y, p):
    pred = p.argmax(axis=1); acc = float(np.mean(pred == y))
    logloss = float(-np.log(np.clip(p[np.arange(len(y)), y], 1e-15, 1.0)).mean())
    conf = p.max(axis=1); correct = pred == y; edges = np.linspace(0, 1, 11); ece = 0.0
    for j in range(10):
        m = (conf >= edges[j]) & (conf < edges[j + 1] if j < 9 else conf <= edges[j + 1])
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return dict(accuracy=acc, logloss=logloss, ece=float(ece))

rows = []
for case in CASES:
    for seed in SEEDS:
        xtr, ytr, xte, yte, xadapt = dataset(case, seed)
        ref = LinearDiscriminantAnalysis(solver="lsqr").fit(xtr, ytr)
        rows.append(dict(case=case, method="sklearn_LDA", seed=seed, **metrics(yte, ref.predict_proba(xte))))
        for variant in VARIANTS:
            model = LDA(variant=variant, random_state=seed)
            if variant == "shift_corrected":
                model.fit(xtr, ytr, X_target=xadapt if xadapt is not None else xtr.copy())
            else:
                model.fit(xtr, ytr)
            rows.append(dict(case=case, method=variant, seed=seed, **metrics(yte, model.predict_proba(xte))))
res = pd.DataFrame(rows)
METHODS = ["sklearn_LDA", *VARIANTS]
mean = res.groupby(["case", "method"])[["accuracy", "logloss", "ece"]].mean()
sd = res.groupby(["case", "method"])[["accuracy", "logloss", "ece"]].std()
def pivot(metric):
    return mean[metric].unstack("method").loc[list(CASES), METHODS].round(4)
print(f"Accuracy (mean over seeds {SEEDS})")
pivot("accuracy")

In [ ]:
print("Log-loss (lower is better)"); print(pivot("logloss").to_string())
print("\nECE (lower is better)"); print(pivot("ece").to_string())
print("\nAccuracy sample SD over seeds"); print(sd["accuracy"].unstack("method").loc[list(CASES), METHODS].round(4).to_string())

In [ ]:
acc = pivot("accuracy")
ax = acc.plot.bar(figsize=(12, 4), width=0.85)
ax.set_ylabel("accuracy"); ax.set_ylim(max(0.0, acc.values.min() - 0.05), 1.0)
ax.set_title("Held-out accuracy by case (mean over seeds)"); plt.xticks(rotation=25)
plt.legend(ncol=3, fontsize=8); plt.tight_layout(); plt.show()

## 4. Look inside each variant

In [ ]:
seed = SEEDS[0]
# shift_corrected: importance weights
xtr, ytr, xte, yte, xadapt = dataset("shifted", seed)
sc = LDA(variant="shift_corrected").fit(xtr, ytr, X_target=xadapt)
w = sc.sample_weights_
print(f"shift_corrected weights: min={w.min():.3f} mean={w.mean():.3f} max={w.max():.3f} (clipped at 20, renormalised)")
# shrinkage intensity
xtr, ytr, *_ = dataset("high_dim", seed)
print("Ledoit-Wolf shrinkage intensity (high_dim):", round(LDA(variant="shrinkage").fit(xtr, ytr).shrinkage_, 3),
      "| (clean):", round(LDA(variant="shrinkage").fit(*dataset("clean", seed)[:2]).shrinkage_, 3))
# sparse: selected features out of 80
sp = LDA(variant="sparse", threshold=1.0).fit(xtr, ytr)
print(f"sparse: {len(sp.selected_features_)} of {xtr.shape[1]} features retained (10 informative): {sp.selected_features_[:15]}")
# heteroscedastic: class covariance traces
xh, yh, *_ = dataset("heteroscedastic", seed)
het = LDA(variant="heteroscedastic").fit(xh, yh)
print("heteroscedastic class-covariance traces:", [round(float(np.trace(S)), 1) for S in het.class_covariances_])

# discriminant projection (wine) with the base variant
xw, yw, xwt, ywt, _ = dataset("wine", seed)
Z = LDA(variant="base").fit(xw, yw).transform(xw)
plt.figure(figsize=(4.8, 3.8)); plt.scatter(Z[:, 0], Z[:, 1], c=yw, cmap="viridis", s=14)
plt.title("StatCore LDA projection (wine)"); plt.show()

## 5. Summary and limitations

In [ ]:
base = mean["accuracy"].unstack("method")
summary = pd.DataFrame({
    "best_method": base[METHODS].idxmax(axis=1),
    "best_acc": base[METHODS].max(axis=1),
    "sklearn_LDA_acc": base["sklearn_LDA"],
    "statcore_base_acc": base["base"]}).loc[list(CASES)].round(4)
print(summary.to_string())

**What to read from the tables.** `base` should match sklearn's LDA closely (same model, different numerical path). `shrinkage` is expected to help when $n\approx d$ (`high_dim`), `sparse` when most features are noise, `heteroscedastic` when class covariances really differ, and `shift_corrected` only when the shift is genuine and the domain classifier is informative.

**Honest limitations**
- In our local 3-seed run: `shift_corrected` was *worse* than plain LDA on the shifted case (accuracy ≈ 0.80 vs 0.87, log-loss ≈ 0.72 vs 0.32) — importance weights with high variance (some > 10 after renormalisation) hurt; and `shrinkage` was clearly worse on the raw (unstandardised) wine features because shrinking towards a scaled identity is scale-sensitive. Standardise features first if you use it.
- `heteroscedastic` wins on wine/heteroscedastic data but collapses when $n\approx d$ (`high_dim`, accuracy ≈ 0.54, log-loss > 12).
- Synthetic Gaussian data favour LDA-type models by construction; wine is small (178 rows), so with a handful of seeds many differences are within noise.
- `shift_corrected` needs unlabeled target data, only corrects *covariate* shift (not label or concept shift), and clips weights at 20, which biases the correction when the shift is severe. In non-shifted cases it is given a copy of the training features (weights ≈ 1), so it only tests that it does no harm.
- `sparse` is a diagonal-covariance model: it ignores feature correlations and its threshold `threshold` is not tuned.
- `heteroscedastic` needs enough samples per class to estimate $\Sigma_k$; its regularisation weight `reg_param` is fixed at 0.3 here.
- Probabilities come from a Gaussian model and are not post-hoc calibrated; ECE is a noisy estimate with 10 bins on a few hundred test points.